# Phase 2 — first end-to-end run (smoke test)
Run the cells **one by one** (Shift+Enter). Goal: prove the pipeline works (GPU → BasicTS → STID on PEMS04 for 2 epochs → one results JSON).
These numbers are a **pipeline check, not a result**: status is saved as `smoke_test`.
Before starting: *Runtime → Change runtime type → T4 GPU*.

## 1. Check GPU

In [ ]:
!nvidia-smi
import sys, torch
print(sys.version)
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

## 2. Mount Google Drive (a pop-up will ask for permission)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone your repo (with the BasicTS submodule)

In [ ]:
%cd /content
!rm -rf proposal
!git clone --recursive https://github.com/hrfpour/proposal.git
%cd /content/proposal
!git log --oneline -3
!ls code/BasicTS | head -30

## 4. Build an isolated Python 3.11 environment for BasicTS
Colab now runs Python 3.13, but BasicTS pins old packages (e.g. numpy 1.24.4) that cannot be built on 3.13.
So we create a separate Python 3.11 environment (`/content/venv`) and install BasicTS' own pinned requirements there.
This takes a few minutes. If you restart the Colab session, run cells 2, 3 and 4 again.

In [ ]:
%cd /content/proposal/code/BasicTS
!pip -q install uv
!uv venv /content/venv --python 3.11
!grep -n -i -E "^(torch|numpy|setuptools|easy)" requirements.txt
!uv pip install --python /content/venv/bin/python -r requirements.txt 2>&1 | tail -n 25

### 4b. Make sure PyTorch exists inside the new environment

In [ ]:
import subprocess
PY = "/content/venv/bin/python"
if subprocess.run([PY, "-c", "import torch"], capture_output=True).returncode != 0:
    print("torch not in requirements.txt -> installing it")
    !uv pip install --python {PY} torch
!{PY} -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__, '| cuda:', torch.cuda.is_available())"

## 5. Show the quick-start lines of YOUR BasicTS version

In [ ]:
%cd /content/proposal/code/BasicTS
!grep -n -i -E "python experiments|unzip|datasets.zip|datasets/" README.md | head -40

## 6. Download datasets.zip straight into Colab (no copy in your Drive needed)
Paste the Google Drive link you copied (File → Share → Copy link) in `DATA_URL`.
If PEMS data is already cached in your Drive from an earlier session, this step is skipped.

In [ ]:
import os
KEEP = ["PEMS04", "PEMS08"]
CACHE = "/content/drive/MyDrive/proposal_data"
LOCAL_ZIP = "/content/datasets.zip"
DATA_URL = "PASTE_YOUR_LINK_HERE"

if all(os.path.isdir(f"{CACHE}/{n}") for n in KEEP):
    print("PEMS data already cached in Drive - nothing to download")
elif os.path.exists(LOCAL_ZIP):
    print("zip already present:", LOCAL_ZIP)
else:
    assert DATA_URL.startswith("http"), "Edit DATA_URL first"
    !pip -q install gdown
    !gdown --fuzzy "{DATA_URL}" -O {LOCAL_ZIP}
!ls -la {LOCAL_ZIP}

## 7. Look inside the zip (only lists names, extracts nothing)

In [ ]:
!unzip -l {LOCAL_ZIP} | grep -i -E "PEMS0[48]" | head -30

## 8. Extract ONLY PEMS04 and PEMS08, and keep a small PEMS-only copy in Drive
Next sessions reuse `MyDrive/proposal_data`, so the big zip is never needed again.

In [ ]:
import os, glob, shutil
BT = "/content/proposal/code/BasicTS"
os.makedirs(f"{BT}/datasets", exist_ok=True)

if not all(os.path.isdir(f"{CACHE}/{n}") for n in KEEP):
    !rm -rf /content/_tmp && mkdir -p /content/_tmp
    !unzip -q -o {LOCAL_ZIP} '*PEMS04*' '*PEMS08*' -d /content/_tmp
    os.makedirs(CACHE, exist_ok=True)
    for n in KEEP:
        hits = [p for p in glob.glob(f"/content/_tmp/**/{n}", recursive=True) if os.path.isdir(p)]
        print(n, "->", hits)
        assert hits, f"Folder {n} not found inside the zip (see cell 7 output)"
        shutil.copytree(hits[0], f"{CACHE}/{n}", dirs_exist_ok=True)

for n in KEEP:
    shutil.copytree(f"{CACHE}/{n}", f"{BT}/datasets/{n}", dirs_exist_ok=True)
!ls -la {BT}/datasets
!ls -la {BT}/datasets/PEMS04

## 9. Find the STID config for PEMS04 and make a 2-epoch copy

In [ ]:
import glob, re, pathlib
%cd /content/proposal/code/BasicTS
cfgs = sorted(glob.glob("baselines/STID/**/*PEMS04*.py", recursive=True))
print("configs found:", cfgs)
src = pathlib.Path(cfgs[0])
dst = src.with_name("PEMS04_smoke.py")
txt = src.read_text()
print("--- lines mentioning epoch in the original config ---")
for i, line in enumerate(txt.splitlines(), 1):
    if "epoch" in line.lower():
        print(i, line)
new, n = re.subn(r"(?i)(num_epochs[\"']?\s*[=:]\s*)\d+", r"\g<1>2", txt)
print("epoch setting replaced:", n, "(should be 1)")
dst.write_text(new)
print("smoke config:", dst)

## 10. Train for 2 epochs (first prints the script's options so we see the right flags)

In [ ]:
PY = "/content/venv/bin/python"
!{PY} experiments/train.py --help 2>&1 | head -30

In [ ]:
PY = "/content/venv/bin/python"
!{PY} experiments/train.py -c {dst} --gpus '0' 2>&1 | tee /content/smoke_run.log | tail -n 60

## 11. Export the results JSON and copy it to Drive

In [ ]:
!python /content/proposal/code/scripts/export_result.py --log-dir checkpoints --model STID --dataset PEMS04 --seed 0 --config {dst} --epochs 2 --status smoke_test --out /content/results_out
!ls /content/results_out
import shutil, time
dest = f"/content/drive/MyDrive/proposal_runs/{time.strftime('%Y%m%d_%H%M')}"
shutil.copytree("/content/results_out", dest)
print("Saved to Drive:", dest)